In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# --- Parameters for VAE_AGE_SITE ---
# !!! Adjust these lists to match your actual experiment parameters !!!
# Define lists of hyperparameters to test
latent_dims = [32, 64]
dropout_values = [0.0]  # VAE dropout

# New separate dropout values for predictors
age_dropout_values = [0.0]
site_dropout_values = [0.0, 0.3]

# Define lists of weight values to test
w_recon_values = [1.0]
w_kl_values = [1.0, 0.1]
w_age_values = [3.0, 5.0]  # Higher weights to prioritize age prediction
w_site_values = [2.0, 3.0]  

df_list = []

folder_path = "csv_files" # Relative path within VAE_AGE_SITE
output_plot_folder = "plots" # Relative path within VAE_AGE_SITE
os.makedirs(output_plot_folder, exist_ok=True) # Create output dir if needed

# Loop over every (latent_dim, vae_dropout, predictor_dropout) combination
print(f"Looking for CSV files in: {os.path.abspath(folder_path)}")
for ld in latent_dims:
    for vae_dr in dropout_values:
        for age_dropout in age_dropout_values:
            for site_dropout in site_dropout_values:
                for w_recon in w_recon_values:
                    for w_kl in w_kl_values:
                        for w_age in w_age_values:
                            for w_site in w_site_values:
                # Filename pattern: vae_combined_metrics_ld{ld}_vae_dr{vae_dr}_pred_dr{pred_dr}.csv
                                csv_file = os.path.join(folder_path, f"combo_ld{ld}_drV{vae_dr}_drA{age_dropout}_drS{site_dropout}_wr{w_recon}_wkl{w_kl}_wa{w_age}_ws{w_site}_metrics.csv")
                                try:
                                    tmp = pd.read_csv(csv_file)
                                    # Add columns for parameters
                                    tmp["latent_dim"] = ld
                                    tmp["vae_dropout"] = vae_dr
                                    tmp["age_dropout"] = age_dropout
                                    tmp["site_dropout"] = site_dropout
                                    tmp["w_recon"] = w_recon
                                    tmp["w_kl"] = w_kl
                                    tmp["w_age"] = w_age
                                    tmp["w_site"] = w_site
                                    df_list.append(tmp)
                                except FileNotFoundError:
                                    print(f"Warning: {csv_file} not found. Skipping.")
                                    continue
                                except pd.errors.EmptyDataError:
                                    print(f"Warning: {csv_file} is empty. Skipping.")
                                    continue
                                except Exception as e:
                                    print(f"Error reading {csv_file}: {e}. Skipping.")
                                    continue

# Combine all into one big DataFrame
if df_list:
    df_all = pd.concat(df_list, ignore_index=True)
    print(f"\nLoaded data for {len(df_list)} combinations.")
    print(f"Total rows: {len(df_all)}")
    print("Columns:", df_all.columns.tolist())
else:
    print("\nWarning: No CSV files found or loaded successfully. Proceeding might cause errors.")
    # Initialize an empty DataFrame with expected columns to avoid downstream errors,
    # although plots will be empty.
    expected_cols = [
        'epoch', 'train_loss', 'val_loss', 'train_recon_loss', 'val_recon_loss',
        'train_kl_loss', 'val_kl_loss', 'train_age_loss', 'val_age_loss',
        'train_site_loss', 'val_site_loss', 'train_age_mae', 'val_age_mae',
        'train_site_acc', 'val_site_acc', 'beta', 'grl_alpha', 'learning_rate',
        'latent_dim', 'vae_dropout', 'predictor_dropout'
    ]
    df_all = pd.DataFrame(columns=expected_cols)


Looking for CSV files in: /Users/samchou/AFQ-Insight-Autoencoder-Plotting/VAE_AGE_SITE/csv_files

Loaded data for 24 combinations.
Total rows: 12000
Columns: ['epoch', 'train_loss', 'val_loss', 'train_recon_loss', 'val_recon_loss', 'train_kl_loss', 'val_kl_loss', 'train_age_loss', 'val_age_loss', 'train_site_loss', 'val_site_loss', 'train_age_mae', 'val_age_mae', 'train_site_acc', 'val_site_acc', 'current_beta', 'current_grl_alpha', 'current_lr', 'latent_dim', 'vae_dropout', 'age_dropout', 'site_dropout', 'w_recon', 'w_kl', 'w_age', 'w_site']


In [2]:
import matplotlib.pyplot as plt
import os
import itertools # Needed for iterating unique combos within groups

# --- Metrics to plot vs epoch (Grouped by w_age and w_site) ---
metrics = [
    "train_loss", "val_loss",
    "train_recon_loss", "val_recon_loss",
    "train_kl_loss", "val_kl_loss",
    "train_age_loss", "val_age_loss",
    "train_site_loss", "val_site_loss",
    "train_age_mae", "val_age_mae",
    "train_site_acc", "val_site_acc"
]

# Make sure the parameter lists from Cell 1 are accessible here
# Ensure these variables are defined (usually from Cell 1)
# w_age_values = [1.0, 3.0]
# w_site_values = [1.0, 3.0]
# output_plot_folder = "plots" # Should be defined in Cell 1

print(f"\nGenerating plots for metrics (grouped by w_age and w_site): {metrics}")
print(f"Saving epoch plots to: {os.path.abspath(output_plot_folder)}")

if not df_all.empty:
    # Ensure 'epoch' column exists
    if 'epoch' not in df_all.columns:
        print("Error: 'epoch' column not found in the combined DataFrame. Cannot generate epoch plots.")
    else:
        for metric in metrics:
            if metric not in df_all.columns:
                print(f"Warning: Metric '{metric}' not found in DataFrame. Skipping group plots.")
                continue

            print(f"\n-- Generating plots for metric: {metric} --")
            # Outer loops for NEW grouping parameters: w_age and w_site
            for w_age in w_age_values:
                 for w_site in w_site_values:

                    plt.figure(figsize=(14, 9))
                    plot_has_data_for_group = False # Flag for this specific plot group

                    # Filter the DataFrame for the current group (fixed w_age, w_site)
                    group_df = df_all[
                        (df_all["w_age"] == w_age) &
                        (df_all["w_site"] == w_site)
                    ]

                    if group_df.empty:
                        # print(f"Info: No data found for metric '{metric}' with W_AGE={w_age}, W_SITE={w_site}. Skipping plot.") # Can be verbose
                        plt.close()
                        continue

                    # Define parameters that vary WITHIN this group
                    varying_params = ['latent_dim', 'vae_dropout', 'predictor_dropout', 'w_recon', 'w_kl']
                    # Ensure varying_params exist in group_df before using them
                    valid_varying_params = [p for p in varying_params if p in group_df.columns]
                    if not valid_varying_params:
                         print(f"Warning: No valid varying parameter columns ({varying_params}) found for group W_AGE={w_age}, W_SITE={w_site}. Skipping plot.")
                         plt.close()
                         continue

                    # Create unique combinations of the varying parameters present in this group
                    unique_combos = group_df[valid_varying_params].drop_duplicates().to_dict('records')

                    for combo in unique_combos:
                        # Dynamically build the filter based on available params in combo
                        combo_filter = (group_df['epoch'] == group_df['epoch']) # Start with a True condition
                        label_parts = []
                        for param, value in combo.items():
                            combo_filter &= (group_df[param] == value)
                            # Abbreviate labels slightly
                            label_abbrevs = {'latent_dim':'LD', 'vae_dropout':'VDR', 'predictor_dropout':'PDR',
                                             'w_recon':'WR', 'w_kl':'WKL', 'w_age':'WA', 'w_site':'WS'}
                            label_parts.append(f"{label_abbrevs.get(param, param)}={value}")

                        subset = group_df[combo_filter]

                        # Check if subset has data and the required columns
                        if not subset.empty and metric in subset.columns and 'epoch' in subset.columns:
                            # Drop rows where the metric might be NaN/inf if needed
                            subset = subset.dropna(subset=['epoch', metric])
                            if not subset.empty:
                                # Construct label for the line showing varying parameters
                                label = ", ".join(label_parts)
                                plt.plot(
                                    subset["epoch"],
                                    subset[metric],
                                    label=label,
                                    alpha=0.8
                                )
                                plot_has_data_for_group = True

                    # After plotting all lines for the current group (w_age, w_site)
                    if plot_has_data_for_group:
                        # Add title indicating the fixed parameters for this plot
                        plt.title(f"{metric} vs. epoch (W_AGE={w_age}, W_SITE={w_site})")
                        plt.xlabel("Epoch")
                        plt.ylabel(metric)
                        plt.grid(True, which='both', linestyle='--', linewidth=0.5)

                        # Adjust legend - based on number of unique combos in this group
                        num_lines_per_plot = len(unique_combos)
                        if num_lines_per_plot > 20: # Adjust threshold if needed
                             plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), ncol=max(1, num_lines_per_plot // 20), fontsize='xx-small') # Adjust ncol, smaller font
                             plt.subplots_adjust(right=0.70, bottom=0.1) # Adjust space for legend
                        elif num_lines_per_plot > 0:
                             plt.legend(fontsize='x-small') # Use smaller font if many params in label

                        # Save the figure with group parameters (wa, ws) in the filename (using output_plot_folder)
                        plot_filename = os.path.join(output_plot_folder, f"{metric}_vs_epoch_wa_{w_age}_ws_{w_site}.png")
                        try:
                             plt.savefig(plot_filename, bbox_inches='tight') # Use bbox_inches='tight' for legend outside plot
                             print(f"Saved plot: {plot_filename}")
                        except Exception as e:
                             print(f"Error saving plot {plot_filename}: {e}")
                        plt.close() # Close the figure for this group
                    else:
                         # Should have been caught by group_df.empty check, but just in case
                         # print(f"Info: Skipping plot for metric '{metric}' with W_AGE={w_age}, W_SITE={w_site} as no valid line data was found.") # Can be verbose
                         plt.close() # Close the empty figure anyway

else:
     # df_all was empty to begin with
     print("Skipping metric vs epoch plots as no data was loaded.")

print("\nFinished generating metric vs epoch plots (grouped by w_age and w_site).")


Generating plots for metrics (grouped by w_age and w_site): ['train_loss', 'val_loss', 'train_recon_loss', 'val_recon_loss', 'train_kl_loss', 'val_kl_loss', 'train_age_loss', 'val_age_loss', 'train_site_loss', 'val_site_loss', 'train_age_mae', 'val_age_mae', 'train_site_acc', 'val_site_acc']
Saving epoch plots to: /Users/samchou/AFQ-Insight-Autoencoder-Plotting/VAE_AGE_SITE/plots

-- Generating plots for metric: train_loss --
Saved plot: plots/train_loss_vs_epoch_wa_3.0_ws_2.0.png
Saved plot: plots/train_loss_vs_epoch_wa_3.0_ws_3.0.png
Saved plot: plots/train_loss_vs_epoch_wa_5.0_ws_2.0.png
Saved plot: plots/train_loss_vs_epoch_wa_5.0_ws_3.0.png

-- Generating plots for metric: val_loss --
Saved plot: plots/val_loss_vs_epoch_wa_3.0_ws_2.0.png
Saved plot: plots/val_loss_vs_epoch_wa_3.0_ws_3.0.png
Saved plot: plots/val_loss_vs_epoch_wa_5.0_ws_2.0.png
Saved plot: plots/val_loss_vs_epoch_wa_5.0_ws_3.0.png

-- Generating plots for metric: train_recon_loss --
Saved plot: plots/train_recon_l

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import numpy as np # Use numpy for nan
import itertools # To iterate through weight combinations

# --- Heatmap Generation (Handling New Weights) ---

# Output folder for heatmaps (defined in Cell 1)
# output_heatmap_folder = "heatmaps"
# Input folder for summaries (defined in Cell 1)
# summary_folder_path = "summaries"

# !!! Ensure these metric names exist in your summary CSV files !!!
heatmap_metrics = {
    "best_val_age_mae": {"cmap": "viridis_r", "fmt": ".3f"}, # Lower is better
    "best_val_site_acc": {"cmap": "viridis", "fmt": ".3f"}   # Higher is better
}

# !!! IMPORTANT: Verify this pattern matches your summary file names !!!
summary_filename_pattern = "combo_ld{ld}_drV{vae_dr}_drP{pred_dr}_wr{w_recon}_wkl{w_kl}_wa{w_age}_ws{w_site}_summary.csv"

# Get parameter lists from Cell 1 (ensure consistency, e.g., predictor_dropout_values)
# These should be defined in Cell 1 or uncomment and define here if running standalone
# latent_dims = [16, 32, 64]
# vae_dropout_values = [0.0]
# predictor_dropout_values = [0.0]
# w_recon_values = [1.0]
# w_kl_values = [1.0, 0.1]
# w_age_values = [1.0, 3.0]
# w_site_values = [1.0, 3.0]

print(f"\nGenerating heatmaps based on summary files in: {os.path.abspath(summary_folder_path)}")
print(f"Saving heatmaps to: {os.path.abspath(output_heatmap_folder)}")
print(f"Using summary file pattern: {summary_filename_pattern}")
print(f"Metrics: {list(heatmap_metrics.keys())}")

# Check if VAE Dropout and Latent Dim actually vary for informative heatmaps
if len(vae_dropout_values) <= 1 and len(latent_dims) <= 1:
    print("\nWarning: Heatmaps show VAE Dropout vs Latent Dimension, but both parameters have only a single value. Heatmaps will be 1x1.")
elif len(vae_dropout_values) <= 1:
     print("\nWarning: Heatmaps show VAE Dropout vs Latent Dimension, but VAE Dropout has only a single value. Heatmaps will have only one row.")
elif len(latent_dims) <= 1:
     print("\nWarning: Heatmaps show VAE Dropout vs Latent Dimension, but Latent Dimension has only a single value. Heatmaps will have only one column.")


for heatmap_metric, plot_opts in heatmap_metrics.items():
    print(f"\n-- Generating heatmaps for {heatmap_metric} --")
    any_heatmap_generated_for_metric = False

    # Define the parameters that are fixed for each heatmap group
    heatmap_grouping_params = [
        predictor_dropout_values,
        w_recon_values,
        w_kl_values,
        w_age_values,
        w_site_values
    ]
    heatmap_grouping_param_names = [ # Corresponding names
        'predictor_dropout', 'w_recon', 'w_kl', 'w_age', 'w_site'
    ]

    # Iterate through all combinations of the grouping parameters
    other_param_combinations = itertools.product(*heatmap_grouping_params)

    for combo_params_tuple in other_param_combinations:
        # Create a dictionary mapping param name to value for this combo
        fixed_params = dict(zip(heatmap_grouping_param_names, combo_params_tuple))
        pred_dr = fixed_params['predictor_dropout'] # Extract for easier use later if needed
        w_recon = fixed_params['w_recon']
        w_kl = fixed_params['w_kl']
        w_age = fixed_params['w_age']
        w_site = fixed_params['w_site']

        # Create an empty DataFrame for this specific combination of other parameters
        # Ensure index/columns are correctly ordered if they have specific meaning
        df_heatmap = pd.DataFrame(index=sorted(vae_dropout_values), columns=sorted(latent_dims), dtype=float)
        df_heatmap.index.name = "VAE Dropout"
        df_heatmap.columns.name = "Latent Dimensions"

        found_summary_data_for_combo = False # Flag specific to this combo
        valid_metric_data_found_for_combo = False # Flag if we found the specific metric column

        # Populate the heatmap for the current fixed combination
        for ld in latent_dims:
            for vae_dr in vae_dropout_values:
                # Construct the summary filename using ALL parameters for this point
                all_params_for_file = {
                    'ld': ld, 'vae_dr': vae_dr, **fixed_params # Combine varying and fixed params
                }
                # Format the filename keys to match the pattern (drV, drP, etc.)
                filename_params_formatted = {
                    'ld': all_params_for_file['ld'],
                    'vae_dr': all_params_for_file['vae_dr'],
                    'pred_dr': all_params_for_file['pred_dr'],
                    'w_recon': all_params_for_file['w_recon'],
                    'w_kl': all_params_for_file['w_kl'],
                    'w_age': all_params_for_file['w_age'],
                    'w_site': all_params_for_file['w_site']
                 }

                summary_file = os.path.join(
                    summary_folder_path,
                    summary_filename_pattern.format(**filename_params_formatted)
                )

                try:
                    tmp = pd.read_csv(summary_file)
                    found_summary_data_for_combo = True

                    if tmp.empty:
                        df_heatmap.loc[vae_dr, ld] = np.nan
                        continue

                    if heatmap_metric in tmp.columns:
                        # Check for non-numeric or NaN values before assigning
                        value = tmp[heatmap_metric].iloc[0]
                        if pd.isna(value):
                             df_heatmap.loc[vae_dr, ld] = np.nan
                        else:
                             try:
                                 df_heatmap.loc[vae_dr, ld] = float(value)
                                 valid_metric_data_found_for_combo = True
                             except (ValueError, TypeError):
                                 print(f"Warning: Non-numeric value '{value}' for {heatmap_metric} in {summary_file}. Setting NaN.")
                                 df_heatmap.loc[vae_dr, ld] = np.nan
                    else:
                        df_heatmap.loc[vae_dr, ld] = np.nan # Metric column doesn't exist

                except FileNotFoundError:
                    df_heatmap.loc[vae_dr, ld] = np.nan # Expected if combo doesn't exist
                except pd.errors.EmptyDataError:
                    # print(f"Warning: Summary file {summary_file} is empty. Setting NaN.") # Can be verbose
                    df_heatmap.loc[vae_dr, ld] = np.nan
                except Exception as e:
                    print(f"Error reading or processing {summary_file}: {e}. Setting NaN.")
                    df_heatmap.loc[vae_dr, ld] = np.nan

        # --- Plotting the Heatmap for this specific combination ---
        # Check if we found any summary files AND if the heatmap isn't entirely NaN
        if found_summary_data_for_combo and not df_heatmap.isnull().all().all():
            # Ensure the DataFrame index and columns are sorted numerically for plotting
            df_heatmap = df_heatmap.sort_index(axis=0).sort_index(axis=1)

            plt.figure(figsize=(max(8, len(latent_dims)*0.8), max(6, len(vae_dropout_values)*0.6 + 1))) # Adjust size
            sns.heatmap(df_heatmap, annot=True, fmt=plot_opts['fmt'], cmap=plot_opts['cmap'], linewidths=.5, cbar_kws={'label': heatmap_metric})

            # Create a descriptive title using the fixed_params dictionary
            title_parts = [f"{name}={val}" for name, val in fixed_params.items()]
            title = f"{heatmap_metric}\n({', '.join(title_parts)})"
            plt.title(title, fontsize=10) # Smaller font for potentially long title
            plt.tight_layout(rect=[0, 0, 1, 0.97]) # Adjust layout to prevent title overlap

            # Create a descriptive filename (using output_heatmap_folder)
            filename_parts = [f"{name.replace('_','').replace('dropout','dr').replace('predictor','P').replace('vae','V').replace('latent','ld').replace('dim','')}{val}" for name, val in fixed_params.items()]
            heatmap_filename = os.path.join(
                output_heatmap_folder, # <<< Save to heatmap folder
                f"{heatmap_metric}_heatmap_{'_'.join(filename_parts)}.png"
            )
            try:
                plt.savefig(heatmap_filename)
                print(f"Saved heatmap: {heatmap_filename}")
                any_heatmap_generated_for_metric = True
            except Exception as e:
                print(f"Error saving heatmap {heatmap_filename}: {e}")
            plt.close()

        # Optional: Add print statements for skipped heatmaps if needed
        # elif not found_summary_data_for_combo:
        #     print(f"Info: Skipping heatmap for combo {fixed_params} as no summary files found.")
        elif found_summary_data_for_combo and df_heatmap.isnull().all().all():
              if not valid_metric_data_found_for_combo:
                   print(f"Warning: Skipping heatmap for combo {fixed_params}. Files found, but none contained valid '{heatmap_metric}' values.")
              else:
                   print(f"Warning: Skipping heatmap for combo {fixed_params} as all found '{heatmap_metric}' values were NaN or invalid.")


    if not any_heatmap_generated_for_metric:
        print(f"Warning: No heatmaps were successfully generated for '{heatmap_metric}'. Check summary files exist, match the pattern '{summary_filename_pattern}', and contain valid data in the '{heatmap_metric}' column.")

print(f"\nFinished generating heatmaps.")

NameError: name 'summary_folder_path' is not defined